### Miguel Baños Baladrón
### Miguel Pérez Francos
### Rodrigo Touceda Tapias

# Importaciones

In [18]:
using CSV, DataFrames, Glob, Statistics, Random, MLJ, CategoricalArrays

# Preparación de los datos (20%)

## 1. Carga y unificación de los datos

In [19]:
base = "Datos Práctica"

# CSV del Investigador A
csv_inv_a = glob("Investigador A/day */*.csv", base)

# CSV del Investigador B
csv_inv_b = glob("Investigador B/*.csv", base)

all_csv = vcat(csv_inv_a, csv_inv_b)

dfs = [CSV.read(file, DataFrame) for file in all_csv]
df_total = vcat(dfs...)

println("Dataset correctamente cargado y unificado.")
println("Número de variables:         ", ncol(df_total))
println("Número de instancias:        ", nrow(df_total))
println("Número de individuos:        ", length(unique(df_total.subject)))
println("Número de clases de salida:  ", length(unique(df_total.Activity)))

Dataset correctamente cargado y unificado.
Número de variables:         563
Número de instancias:        10299
Número de individuos:        30
Número de clases de salida:  6


In [20]:
println("Clases:\n")
for activity in unique(df_total.Activity)
    println(activity)
end

Clases:

STANDING
SITTING
LAYING
WALKING
WALKING_UPSTAIRS
WALKING_DOWNSTAIRS


## 2. Análisis de valores ausentes

In [21]:
# Porcentajes de nulos por variable
nulos_por_variable = DataFrame(
    Variable = names(df_total),
    PorcentajeNulos = [count(ismissing, df_total[!, col]) / nrow(df_total) * 100 for col in names(df_total)]
)

# Porcentaje de nulos en todo el dataset
total_nulos = sum(count(ismissing, df_total[!, col]) for col in names(df_total))
total_valores = nrow(df_total) * ncol(df_total)

porcentaje_total_nulos = (total_nulos / total_valores) * 100

println("Porcentaje total de valores nulos en el dataset: $(porcentaje_total_nulos)%")

Porcentaje total de valores nulos en el dataset: 0.9984242033534787%


## 3. Tratamiento y transformación de datos

In [22]:
n = nrow(df_total)

res = DataFrame(variable = String[], n_missing = Int[], porcentaje = Float64[])

for col in names(df_total)
    n_miss = count(ismissing, df_total[!, col])
    porc = round((n_miss / n) * 100, digits=2)
    push!(res, (string(col), n_miss, porc))
end

# Ordenar de mayor a menor porcentaje
sort!(res, :porcentaje, rev=true)

# Mostrar solo las 10 primeras
first(res, 3)

Row,variable,n_missing,porcentaje
,String,Int64,Float64
1,tBodyGyroMag-mad(),1033,10.03
2,tBodyGyroMag-iqr(),1033,10.03
3,fBodyAcc-mad()-Y,1032,10.02


- ### Como la variable con mayor porcentaje de nulos no presenta un valor muy alto, decidimos imputar todas las variables

In [23]:
# Rellenamos con la mediana los valores nulos porque es menos sensible a outliers
df_imputado = deepcopy(df_total)
gdf = groupby(df_imputado, :subject) # Agrupamos por individuo para que cada dato nulo se rellene con la mediana de los valores de dicho individuo.

for subdf in gdf
    for col in names(subdf)
        if col in (:subject, :Activity)
            continue
        end

        # Ignoramos variables no numéricas
        coldata = subdf[!, col]
        if !(eltype(skipmissing(coldata)) <: Number)
            continue
        end

        mediana = median(skipmissing(coldata))
        replace!(coldata, missing => mediana)
    end
end

## 4. Partición Holdout

In [24]:
# Lista de sujetos únicos
subjects = unique(df_imputado.subject)

# Fijamos semilla 104
Random.seed!(104)

# Barajamos el orden de los sujetos
shuffle!(subjects)

# 10% de sujetos para test
n_test = round(Int, length(subjects) * 0.10)

test_subjects      = subjects[1:n_test]
trainval_subjects  = subjects[n_test+1:end]

println("Sujetos en TEST: ", sort(test_subjects))
println("Sujetos en TRAIN+VAL: ", sort(trainval_subjects))

df_trainval = filter(row -> row.subject in trainval_subjects, df_imputado); 
df_test     = filter(row -> row.subject in test_subjects, df_imputado);

Sujetos en TEST: [18, 22, 25]
Sujetos en TRAIN+VAL: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 21, 23, 24, 26, 27, 28, 29, 30]


## 5. Cross-Validation individual wise
## 6. Normalización Min-Max

In [25]:
features_cols = names(df_trainval, Not([:subject, :Activity]))
X = Matrix(df_trainval[:, features_cols])
y = df_trainval.Activity
groups = df_trainval.subject

Random.seed!(104)

unique_subjects = unique(groups)
shuffled = shuffle(unique_subjects)
n_subj = length(unique_subjects)

k = 5
# Crear folds 
fold_ids = [mod1(i,k) for i in 1:n_subj] # mod1 para que los índices vayande 1 a k 
folds_val_subjects = [shuffled[fold_ids .== f] for f in 1:k] # Lista de sujetos por fold

for (i, val_subj_fold) in enumerate(folds_val_subjects)
    println("\n=== Fold $i ===")

    # Índices de validación y entrenamiento
    val_idx  = findall(x -> x in val_subj_fold, groups)
    train_idx = findall(x -> !(x in val_subj_fold), groups)

    # Normalización Min-Max 
    x_train = X[train_idx,:]
    x_val = X[val_idx,:]
    min = minimum.(eachcol(x_train))
    max = maximum.(eachcol(x_train))
    num_cols = size(x_train, 2)
    for j in 1:num_cols
        range = max[j] - min[j]

        # Evitar división por cero → dividir por 1
        if range == 0
            range = 1.0
        end

        x_train[:, j] = (x_train[:, j] .- min[j]) ./ range
        x_val[:, j]   = (x_val[:, j] .- min[j]) ./ range
    end
    
    # Mostrar info útil
    println("Sujetos en VALIDACIÓN: ", sort(val_subj_fold))
    println("Nº instancias TRAIN: ", length(train_idx))
    println("Nº instancias VAL:   ", length(val_idx))
end


=== Fold 1 ===
Sujetos en VALIDACIÓN: [11, 12, 15, 16, 20, 23]
Nº instancias TRAIN: 7149
Nº instancias VAL:   2056

=== Fold 2 ===
Sujetos en VALIDACIÓN: [1, 4, 13, 21, 24, 27]
Nº instancias TRAIN: 7049
Nº instancias VAL:   2156

=== Fold 3 ===
Sujetos en VALIDACIÓN: [2, 3, 8, 10, 28]
Nº instancias TRAIN: 7605
Nº instancias VAL:   1600

=== Fold 4 ===
Sujetos en VALIDACIÓN: [6, 7, 14, 19, 26]
Nº instancias TRAIN: 7497
Nº instancias VAL:   1708

=== Fold 5 ===
Sujetos en VALIDACIÓN: [5, 9, 17, 29, 30]
Nº instancias TRAIN: 7520
Nº instancias VAL:   1685


In [26]:
# Normalización Min-Max para el conjunto TEST 

X_trainval = Matrix(df_trainval[:, features_cols])
X_test = Matrix(df_test[:, features_cols])

min_vals = minimum.(eachcol(X_trainval))
max_vals = maximum.(eachcol(X_trainval))

# 3. Escalar trainval y test usando esos min/max
X_trainval_scaled = copy(X_trainval)
X_test_scaled = copy(X_test)

num_cols = size(X_trainval, 2)

for j in 1:num_cols
    range = max_vals[j] - min_vals[j]
    if range == 0
        range = 1.0
    end

    X_trainval_scaled[:, j] = (X_trainval[:, j] .- min_vals[j]) ./ range
    X_test_scaled[:, j]     = (X_test[:, j]    .- min_vals[j]) ./ range
end

println("\n=== Conjunto TEST ===")
println("Sujetos en TEST: ", sort(test_subjects))
println("Nº instancias TEST: ", nrow(df_test))


=== Conjunto TEST ===
Sujetos en TEST: [18, 22, 25]
Nº instancias TEST: 1094


# Modelos básicos y selección de atributos (20%)

- ### Filtrado ANOVA

In [27]:
feature_cols = names(df_trainval, Not([:subject, :Activity]))

# Cálculo del ANOVA F-score para una columna (feature)
function anova(column_name, df_trainval)
    x = df_trainval[!, column_name] # Vector completo de la feature
    groups = groupby(df_trainval, :Activity) # Agrupar por clase
    mean_global = mean(x) # Media global
    k = length(groups)
    N = nrow(df_trainval)
    SST = sum(nrow(g) * (mean(g[!, column_name]) - mean_global)^2 for g in groups)
    SSE = sum(sum((g[!, column_name] .- mean(g[!, column_name])).^2) for g in groups)
    F = (SST / (k - 1)) / (SSE / (N - k))
    return F
end

# Calcular F-score de cada columna usando tu función ANOVA
F_scores = Dict{String, Float64}()

for col in feature_cols
    F_scores[col] = anova(col, df_trainval)
end

# Seleccionar el 10% de columnas con mayor F-score
num_features = length(feature_cols)
num_select = max(1, Int(ceil(0.10 * num_features)))
sorted_pairs = sort(collect(F_scores), by = x -> x[2], rev = true) # Ordenar de mayor a menor los F-scores
top_features = sorted_pairs[1:num_select]
selected_cols = first.(top_features)

println("Seleccionadas $(length(selected_cols)) columnas del 10%:")
println(selected_cols)

Seleccionadas 57 columnas del 10%:
["fBodyAccJerk-entropy()-X", "tGravityAcc-mean()-X", "tGravityAcc-min()-X", "fBodyAccJerk-entropy()-Y", "tGravityAcc-energy()-X", "fBodyBodyAccJerkMag-entropy()", "fBodyAcc-entropy()-X", "tBodyAcc-max()-X", "tBodyAccJerkMag-entropy()", "tBodyAccMag-mean()", "tGravityAccMag-mean()", "tBodyAcc-sma()", "tBodyAccJerk-entropy()-X", "tGravityAccMag-sma()", "tBodyAccMag-sma()", "tBodyAcc-std()-X", "fBodyAcc-sma()", "fBodyAcc-mad()-X", "fBodyAccJerk-entropy()-Z", "tBodyAccJerk-entropy()-Z", "fBodyAcc-mean()-X", "tBodyAcc-mad()-X", "angle(X,gravityMean)", "tBodyAccJerkMag-sma()", "tBodyAccJerkMag-mean()", "tBodyAccJerk-sma()", "tBodyAccJerkMag-mad()", "fBodyAccJerk-sma()", "fBodyAcc-entropy()-Y", "fBodyAccMag-entropy()", "tGravityAccMag-max()", "fBodyAccMag-mean()", "tBodyAccJerk-std()-X", "tBodyAccJerkMag-std()", "fBodyAccMag-sma()", "tBodyAccJerk-mad()-X", "tBodyAccJerkMag-iqr()", "fBodyAccMag-mad()", "fBodyAcc-entropy()-Z", "fBodyBodyAccJerkMag-sma()", "fBo

- ### Filtrado de Pearson

In [29]:
# Cálculo de la correlación de Pearson entre dos variables
function pearson(x, y)
    mean_x = mean(x)
    mean_y = mean(y)
    num = sum((x .- mean_x) .* (y .- mean_y))
    den = sqrt(sum((x .- mean_x).^2) * sum((y .- mean_y).^2))
    return den == 0 ? 0.0 : num / den
end

# Convertimos la etiqueta a numérica
classes = unique(df_trainval.Activity)
class_to_int = Dict(c => i for (i, c) in enumerate(classes))
y_num = Float64.(map(c -> class_to_int[c], df_trainval.Activity))


# Correlación de Pearson entre cada feature y la variable objetivo
correlations = Dict{String, Float64}()

for col in feature_cols
    x = df_trainval[!, col]
    correlations[col] = abs(pearson(x, y_num))
end

# Seleccionamos el 10% con mayor |Pearson|
num_features = length(feature_cols)
num_select = max(1, Int(ceil(0.10 * num_features)))

sorted_corr = sort(collect(correlations), by = x -> x[2], rev = true)
top_features_corr = sorted_corr[1:num_select]

selected_cols_pearson = first.(top_features_corr)

println("Seleccionadas $(length(selected_cols_pearson)) columnas (10%) según |Pearson|:")
println(selected_cols_pearson)

Seleccionadas 57 columnas (10%) según |Pearson|:
["tBodyAcc-std()-X", "tBodyAcc-max()-X", "tBodyAcc-mad()-X", "fBodyAcc-mad()-X", "fBodyAcc-mean()-X", "tGravityAccMag-mean()", "tBodyAccMag-mean()", "tGravityAccMag-sma()", "tBodyAccMag-sma()", "fBodyAccMag-mad()", "tBodyAcc-sma()", "fBodyAcc-max()-X", "tGravityAccMag-max()", "tBodyAccMag-std()", "fBodyAccMag-mean()", "fBodyAccMag-sma()", "fBodyAcc-sma()", "tGravityAccMag-mad()", "tBodyAccMag-mad()", "fBodyAccMag-std()", "fBodyAcc-entropy()-X", "fBodyAccJerk-entropy()-X", "fBodyAccMag-iqr()", "tBodyAcc-iqr()-X", "fBodyBodyAccJerkMag-entropy()", "tBodyAccJerk-std()-X", "tBodyAccJerk-mad()-X", "tBodyAcc-min()-X", "fBodyAccJerk-mad()-X", "fBodyAccJerk-std()-X", "tGravityAccMag-energy()", "tBodyAccMag-energy()", "fBodyAccJerk-sma()", "fBodyBodyAccJerkMag-sma()", "fBodyBodyAccJerkMag-mean()", "fBodyAccMag-entropy()", "fBodyAccJerk-entropy()-Z", "tBodyAccJerk-iqr()-X", "tBodyAccJerkMag-sma()", "tBodyAccJerkMag-mean()", "tBodyAccJerk-sma()", "t

- ### Filtrado de Spearman

In [32]:
function rank_vector(v)
    order = sortperm(v) # Obtiene los índices que ordenarían el vector
    ranks = similar(v, Float64) # Vector de rangos vacío

    # Asignar rangos según posición en el orden
    for i in 1:length(v)
        ranks[order[i]] = i
    end
    return ranks
end

function spearman(x, y)
    rank_x = rank_vector(x)
    rank_y = rank_vector(y)
    return pearson(rank_x, rank_y)
end

# Convertir Activity a numérica
classes = unique(df_trainval.Activity)
class_to_int = Dict(c => i for (i, c) in enumerate(classes))
y_num = Float64.(map(c -> class_to_int[c], df_trainval.Activity))

# Diccionario de correlaciones
spearman_scores = Dict{String, Float64}()

for col in feature_cols
    x = df_trainval[!, col]
    spearman_scores[col] = abs(spearman(x, y_num))
end

# Seleccionar el 10% superior
num_features = length(feature_cols)
num_select = max(1, Int(ceil(0.10 * num_features)))

sorted_spearman = sort(collect(spearman_scores), by = x -> x[2], rev = true)
selected_cols_spearman = first.(sorted_spearman[1:num_select])

println("Seleccionadas $(length(selected_cols_spearman)) columnas (10%) según Spearman:")
println(selected_cols_spearman)

Seleccionadas 57 columnas (10%) según Spearman:
["fBodyAcc-bandsEnergy()-1,8", "fBodyAcc-max()-X", "fBodyAcc-bandsEnergy()-1,16", "fBodyAcc-bandsEnergy()-1,24", "tBodyAcc-energy()-X", "fBodyAcc-energy()-X", "tBodyAcc-std()-X", "tBodyAcc-mad()-X", "tBodyAcc-max()-X", "tBodyAcc-iqr()-X", "fBodyAcc-mad()-X", "fBodyAcc-mean()-X", "fBodyAcc-entropy()-X", "fBodyAccJerk-bandsEnergy()-1,16", "fBodyAcc-std()-X", "tBodyAcc-min()-X", "fBodyAcc-bandsEnergy()-9,16", "fBodyAccJerk-entropy()-X", "fBodyAccJerk-bandsEnergy()-1,8", "fBodyAcc-bandsEnergy()-41,48", "tBodyAccJerk-max()-X", "fBodyAcc-bandsEnergy()-33,48", "fBodyBodyAccJerkMag-entropy()", "fBodyAccMag-mad()", "fBodyAccJerk-energy()-X", "tBodyAccJerk-std()-X", "fBodyAccMag-std()", "fBodyAccJerk-bandsEnergy()-9,16", "tBodyAccJerk-mad()-X", "tBodyAccMag-std()", "tGravityAccMag-max()", "fBodyAccMag-energy()", "tGravityAccMag-energy()", "tBodyAccMag-energy()", "fBodyAcc-iqr()-X", "fBodyAcc-bandsEnergy()-33,40", "fBodyAccJerk-bandsEnergy()-33,48",

- ### Filtrado de Kendall Tau

In [33]:
# Cálculo de la correlación de Kendall entre dos variables
function kendall(x, y)
    n = length(x)
    num_concordant = 0
    num_discordant = 0

    for i in 1:n-1
        for j in i+1:n
            if (x[i] - x[j]) * (y[i] - y[j]) > 0
                num_concordant += 1
            elseif (x[i] - x[j]) * (y[i] - y[j]) < 0
                num_discordant += 1
            end
        end
    end

    num = num_concordant - num_discordant
    return num / (1/2 * n * (n -1))
end

# Convertir Activity a numérica
classes = unique(df_trainval.Activity)
class_to_int = Dict(c => i for (i, c) in enumerate(classes))
y_num = Float64.(map(c -> class_to_int[c], df_trainval.Activity))

# Diccionario de correlaciones
kendall_scores = Dict{String, Float64}()

for col in feature_cols
    x = df_trainval[!, col]
    kendall_scores[col] = abs(kendall(x, y_num))
end

# Seleccionar el 10% superior
num_features = length(feature_cols)
num_select = max(1, Int(ceil(0.10 * num_features)))

sorted_kendall = sort(collect(kendall_scores), by = x -> x[2], rev = true)
selected_cols_kendall = first.(sorted_kendall[1:num_select])

println("Seleccionadas $(length(selected_cols_kendall)) columnas (10%) según Kendall:")
println(selected_cols_kendall)

Seleccionadas 57 columnas (10%) según Kendall:
["fBodyAcc-bandsEnergy()-1,8", "fBodyAcc-max()-X", "fBodyAcc-bandsEnergy()-1,16", "fBodyAcc-bandsEnergy()-1,24", "tBodyAcc-energy()-X", "fBodyAcc-energy()-X", "tBodyAcc-std()-X", "tBodyAcc-mad()-X", "tBodyAcc-max()-X", "fBodyAcc-mad()-X", "tBodyAcc-iqr()-X", "fBodyAcc-mean()-X", "fBodyAcc-std()-X", "fBodyAccJerk-bandsEnergy()-1,16", "tBodyAcc-min()-X", "fBodyAcc-entropy()-X", "fBodyAccJerk-bandsEnergy()-1,8", "fBodyAccMag-std()", "fBodyAcc-bandsEnergy()-9,16", "fBodyAccMag-mad()", "tBodyAccMag-std()", "fBodyAccMag-energy()", "tGravityAccMag-mad()", "tBodyAccMag-mad()", "tGravityAccMag-energy()", "tBodyAccMag-energy()", "tGravityAccMag-max()", "tBodyAccJerk-max()-X", "fBodyAcc-bandsEnergy()-33,48", "fBodyAcc-bandsEnergy()-41,48", "tBodyAcc-sma()", "fBodyAccJerk-energy()-X", "tBodyAccJerk-std()-X", "tGravityAccMag-mean()", "tBodyAccMag-sma()", "tBodyAccMag-mean()", "tGravityAccMag-sma()", "tBodyAccJerk-mad()-X", "fBodyAccMag-sma()", "fBodyAc

- ### Filtrado Mutual Information

- ### Filtrado RFE